In [2]:
import os
from shiny import App, ui, render
from shiny.ui import tags
from PIL import Image
import requests


In [9]:
# Github access
GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")
headers = {"Authorization": f"token {GITHUB_TOKEN}"}

In [10]:
# Define the base URL for raw GitHub content
base_url = "https://raw.githubusercontent.com/Zephyr-Sylvester/circumpolar-connectivity-analysis/master/fig-lib"

# List of folders to explore
folders = ["mapping", "transport-by-ng", "spawning-locations", "retention-sensitivity"]


In [13]:
def get_image_urls_recursive(folder_path):
    """Recursively fetch all PNG image URLs from a folder and its subfolders."""
    urls = []
    api_url = f"https://api.github.com/repos/Zephyr-Sylvester/circumpolar-connectivity-analysis/contents/{folder_path}"

    # Make the API request with authentication
    headers = {"Authorization": f"token {GITHUB_TOKEN}"}
    response = requests.get(api_url, headers=headers)

    if response.status_code == 200:
        items = response.json()
        for item in items:
            if item["type"] == "file" and item["name"].endswith(".png"):
                # Build the raw URL to access the image
                image_url = f"{base_url}/{folder_path.split('/', 1)[1]}/{item['name']}"
                urls.append(image_url)
            elif item["type"] == "dir":
                # If it’s a directory, recurse into it
                urls.extend(get_image_urls_recursive(f"{folder_path}/{item['name']}"))
    else:
        print(f"Failed to fetch contents from {folder_path}: {response.status_code}")
    return urls


In [14]:

# Fetch all image URLs from the main folders and their subfolders
image_urls = []
for folder in folders:
    image_urls.extend(get_image_urls_recursive(f"fig-lib/{folder}"))


Failed to fetch contents from fig-lib/mapping: 401
Failed to fetch contents from fig-lib/transport-by-ng: 401
Failed to fetch contents from fig-lib/spawning-locations: 401
Failed to fetch contents from fig-lib/retention-sensitivity: 401


In [ ]:

# Define the Shiny app UI
app_ui = ui.page_fluid(
    ui.h1("Krill Movement Plot Gallery"),
    ui.row(
        *[
            ui.column(4, tags.img(src=image_url, style="width:100%; margin-bottom: 10px;"))
            for image_url in image_urls
        ]
    )
)

# Define the server logic (empty since it’s a static gallery)
def server(input, output, session):
    pass

# Create the Shiny app
app = App(app_ui, server)